# 03 — Exploratory Data Analysis

Explores the fully-featured dataset (output of `02_feature_engineering.ipynb`) to understand what we're actually searching over: how good the data is, how it's distributed, and what that implies for the search experience.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_pickle('../game_on/data/df_clean.pkl')
df.shape

## 2. Overview

In [ ]:
df.info()

## 3. Quality score distribution

If this skews too high or too low, the 0.55/0.45 match/quality blend used later in scoring would need rebalancing.

In [ ]:
print(df['quality_score'].describe())

plt.figure(figsize=(10, 4))
plt.hist(df['quality_score'].dropna(), bins=40, color='steelblue', edgecolor='black')
plt.title('Quality score distribution')
plt.xlabel('quality_score')
plt.ylabel('Number of games')
plt.show()

## 4. Most common genres

In [ ]:
df['genre'].str.replace('Game genre: ', '', regex=False) \
    .str.split(', ') \
    .explode() \
    .value_counts() \
    .head(15) \
    .plot(kind='barh', figsize=(8, 6), color='steelblue') \
    .invert_yaxis()

## 5. How rich are the descriptions?

The embedding is built mostly from `game_description`. A game with a 3-word description gives SBERT much less to work with than one with a real paragraph — worth knowing which games fall into that bucket.

In [ ]:
df['description_word_count'] = df['game_description'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

print(df['description_word_count'].describe())

plt.figure(figsize=(10, 4))
plt.hist(df['description_word_count'], bins=50, color='steelblue', edgecolor='black')
plt.axvline(df['description_word_count'].mean(), color='red', linestyle='--', label='mean')
plt.axvline(df['description_word_count'].median(), color='green', linestyle='--', label='median')
plt.title('Words per game_description')
plt.xlabel('Word count')
plt.ylabel('Number of games')
plt.legend()
plt.show()

## 6. Price distribution

In [ ]:
print((df['original_price'] == 0).mean(), '% of games are free')

plt.figure(figsize=(10, 4))
plt.hist(df[df['original_price'] > 0]['original_price'], bins=50, color='steelblue', edgecolor='black')
plt.title('Price distribution (paid games only)')
plt.xlabel('Price (USD)')
plt.ylabel('Number of games')
plt.show()

## 7. Games released per year

In [ ]:
df['release_date'].value_counts().sort_index().plot(
    kind='bar', figsize=(12, 4), color='steelblue'
)
plt.title('Games released per year')
plt.xlabel('Year')
plt.ylabel('Number of games')
plt.show()

## 8. Top games by quality score

In [ ]:
df.sort_values('quality_score', ascending=False)[['name', 'quality_score', 'genre']].head(15)